In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

def plot_stock_1min_ohlcv(ymd, stock_id):
    # 1. 構建檔案路徑
    file_path = f"C:/Users/user/Documents/_04_lupFinmind/1min/{ymd}/{stock_id}.parquet"
    

    # 2. 讀取 Parquet
    df = pd.read_parquet(file_path)
    
    # 3. 處理時間：將 date 與 minute 合併為 datetime 物件
    # 這樣 Plotly 才能正確識別時間軸
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['minute'])
    df = df.sort_values('datetime')

    # 4. 建立圖表 (包含上下兩個子圖：K線與成交量)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                       vertical_spacing=0.03, subplot_titles=(f'{stock_id} - {ymd} 1min K', 'Volume'), 
                       row_width=[0.2, 0.7])

    # 5. 加入 K線圖 (Candlestick)
    fig.add_trace(go.Candlestick(
        x=df['datetime'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='OHLC'
    ), row=1, col=1)

    # 6. 加入成交量圖 (Bar)
    # 根據收盤價漲跌給予成交量顏色 (紅/綠)
    colors = ['red' if row['close'] >= row['open'] else 'green' for _, row in df.iterrows()]
    
    fig.add_trace(go.Bar(
        x=df['datetime'],
        y=df['volume'],
        marker_color=colors,
        name='Volume'
    ), row=2, col=1)

    # 7. 圖表設定
    fig.update_layout(
        title=f"Stock {stock_id} Intraday Analysis ({ymd})",
        yaxis_title="Price",
        yaxis2_title="Volume",
        xaxis_rangeslider_visible=False, # 關閉下方滑桿，讓畫面更簡潔
        height=800,
        template="plotly_dark" # 使用深色主題，視覺效果較好
    )

    # 8. 顯示圖表
    fig.show()


In [12]:
plot_stock_1min_ohlcv("2024-12-30", "2345")